In [22]:
import pandas as pd
import numpy as np

#### 1. LOAD RAW CSV FILE

In [23]:

orders_df = pd.read_csv("raw_data/orders_and_shipments.csv", encoding="latin1")


#### 2. FUNCTION TO GENERATE DATA PROFILES

In [24]:

def profile_df(df, name):
    """
    Generates a quick profile: data types, missing values, % missing, and unique counts
    """
    profile = pd.DataFrame({
        "Data Type": df.dtypes,
        "Missing Count": df.isna().sum(),
        "Missing %": (df.isna().mean() * 100).round(2),
        "Unique Count": df.nunique()
    })
    print(f"\n{name} PROFILE")
    display(profile)
    return profile

# ----------------------------
# Raw Profile
# ----------------------------
raw_orders_profile = profile_df(orders_df, "ORDERS (RAW)")



ORDERS (RAW) PROFILE


,Data Type,Missing Count,Missing %,Unique Count
Order ID,int64,0,0.0,11072
Order Item ID,int64,0,0.0,30871
Order YearMonth,int64,0,0.0,36
Order Year,int64,0,0.0,3
Order Month,int64,0,0.0,12
Order Day,int64,0,0.0,31
Order Time,object,0,0.0,1440
Order Quantity,int64,0,0.0,5
Product Department,object,0,0.0,11
Product Category,object,0,0.0,49


#### 3. CLEAN COLUMN NAMES

In [25]:

def clean_columns(df):
    """
    Standardizes column names: lowercase, remove leading/trailing spaces,
    replace spaces with underscores, replace % with pct
    """
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace("%", "pct")
    )
    return df

orders_df = clean_columns(orders_df)

In [26]:
print(orders_df.columns)

Index(['order_id', 'order_item_id', 'order_yearmonth', 'order_year',
       'order_month', 'order_day', 'order_time', 'order_quantity',
       'product_department', 'product_category', 'product_name', 'customer_id',
       'customer_market', 'customer_region', 'customer_country',
       'warehouse_country', 'shipment_year', 'shipment_month', 'shipment_day',
       'shipment_mode', 'shipment_days_-_scheduled', 'gross_sales',
       'discount_pct', 'profit'],
      dtype='object')


#### 4. CLEAN PRODUCT NAMES

In [27]:
def clean_product_name(df):
    df["product_name"] = df["product_name"].str.strip().str.lower()
    return df

orders_df = clean_product_name(orders_df)

#### 5. DATA TYPE CONVERSIONS & STRUCTURAL CLEANING

In [28]:

# ----- ORDERS -----
# discount_pct may have missing or object values → convert to numeric
orders_df["discount_pct"] = pd.to_numeric(orders_df["discount_pct"], errors="coerce")

# Convert order_time to proper datetime.time objects
orders_df["order_time"] = pd.to_datetime(
    orders_df["order_time"], format="%H:%M", errors="coerce"
).dt.time

# Create proper order_date column from year, month, day
orders_df["order_date"] = pd.to_datetime({
    "year": orders_df["order_year"],
    "month": orders_df["order_month"],
    "day": orders_df["order_day"]
})



# ----------------------------
# Post-Conversion Profile (highlight missing values)
# ----------------------------
post_conversion_orders_profile = profile_df(orders_df, "ORDERS (POST-CONVERSION)")



ORDERS (POST-CONVERSION) PROFILE


,Data Type,Missing Count,Missing %,Unique Count
order_id,int64,0,0.00,11072
order_item_id,int64,0,0.00,30871
order_yearmonth,int64,0,0.00,36
order_year,int64,0,0.00,3
order_month,int64,0,0.00,12
order_day,int64,0,0.00,31
order_time,object,0,0.00,1440
order_quantity,int64,0,0.00,5
product_department,object,0,0.00,11
product_category,object,0,0.00,49



#### 6. HANDLE MISSING VALUES
Discount_pct has 1749 missing values: fill missing with 0 instead of average
Rationale: Missing discount likely means no discount applied. Using 0 preserves accuracy
No missing values expected in inventory or fulfillment numeric columns

In [29]:

orders_df["discount_pct"] = orders_df["discount_pct"].fillna(0)

# ----------------------------
# Final Cleaned Profile
# ----------------------------
clean_orders_profile = profile_df(orders_df, "ORDERS (CLEANED)")


ORDERS (CLEANED) PROFILE


,Data Type,Missing Count,Missing %,Unique Count
order_id,int64,0,0.0,11072
order_item_id,int64,0,0.0,30871
order_yearmonth,int64,0,0.0,36
order_year,int64,0,0.0,3
order_month,int64,0,0.0,12
order_day,int64,0,0.0,31
order_time,object,0,0.0,1440
order_quantity,int64,0,0.0,5
product_department,object,0,0.0,11
product_category,object,0,0.0,49



#### 7. MIN/MAX CHECKS FOR NUMERIC COLUMNS
We check for outliers or impossible values

In [30]:
print("\n--- ORDERS: MIN/MAX CHECK ---")
print(orders_df[["order_quantity", "gross_sales", "profit", "discount_pct"]].describe())



--- ORDERS: MIN/MAX CHECK ---
       order_quantity   gross_sales        profit  discount_pct
count    30871.000000  30871.000000  30871.000000  30871.000000
mean         2.149817    200.235690    107.463445      0.101296
std          1.461393    114.251482     61.814582      0.070423
min          1.000000     10.000000      1.000000      0.000000
25%          1.000000    120.000000     65.000000      0.040000
50%          1.000000    200.000000    100.000000      0.090000
75%          3.000000    300.000000    147.000000      0.160000
max          5.000000    533.000000    258.000000      0.250000


#### 8. EXPORT CLEAN CSVs FOR TABLEAU

In [31]:

orders_df.to_csv("clean_data/clean_orders.csv", index=False)